In [1]:
import lasio
import pandas as pd
import os

In [2]:
path = "/Users/wanbalqish/Downloads/Geohackathon data/Force_2020_all_wells_train_test_blind_hidden_final/15_9-13.las"
las = lasio.read(path)

In [5]:
print('Well name:', las.well.WELL.value)
print('Number of curves:', len(las.curves))
print('Number of samples:', len(las.data))
print('\nFirst few rows:')
print(las.df().head())

Well name: 15/9-13 Sleipner East Appr
Number of curves: 21
Number of samples: 21441

First few rows:
        FORCE_2020_LITHOFACIES_CONFIDENCE  FORCE_2020_LITHOFACIES_LITHOLOGY  \
DEPT                                                                          
25.000                                NaN                               NaN   
25.152                                NaN                               NaN   
25.304                                NaN                               NaN   
25.456                                NaN                               NaN   
25.608                                NaN                               NaN   

        CALI  MUDWEIGHT  ROP  RDEP  RSHA  RMED  RXO  SP         DTC  NPHI  \
DEPT                                                                        
25.000   NaN        NaN  NaN   NaN   NaN   NaN  NaN NaN  206.224609   NaN   
25.152   NaN        NaN  NaN   NaN   NaN   NaN  NaN NaN  206.224625   NaN   
25.304   NaN        NaN  NaN   NaN   

In [6]:
las_folder = "/Users/wanbalqish/Downloads/Geohackathon data/Force_2020_all_wells_train_test_blind_hidden_final/"
las_files = [f for f in os.listdir(las_folder) if f.endswith('.las')]

In [7]:
print(f'Total .las files: {len(las_files)}')
print('\nFirst 10 files:')
for f in sorted(las_files)[:10]:
    print(f'  {f}')

Total .las files: 118

First 10 files:
  15_9-13.las
  15_9-14.las
  15_9-15.las
  15_9-17.las
  15_9-23.las
  16_1-2.las
  16_1-6 A.las
  16_10-1.las
  16_10-2.las
  16_10-3.las


In [8]:
# read each las file, extract DEPTH + log curves + WELL NAME, combine into df, save as csv

from pathlib import Path

las_folder = "/Users/wanbalqish/Downloads/Geohackathon data/Force_2020_all_wells_train_test_blind_hidden_final/"

In [9]:
REQUIRED_CURVES = [
    'GR', 'RHOB', 'NPHI', 'PEF', 'DTC', 'DTS',
    'RDEP', 'RSHA', 'RMED', 'ROED',
    'FORMATION', 'GROUP'
]

In [10]:
las_files = sorted([f for f in os.listdir(las_folder) if f.endswith('.las')])

all_data = []
errors = []

In [11]:
for i, las_file in enumerate(las_files):
    try:
        las_path = os.path.join(las_folder, las_file)
        las = lasio.read(las_path)

        df = las.df()

        well_name = las.well.WELL.value
        df['WELL'] = well_name

        df['DEPTH'] = df.index

        all_data.append(df)

        if (i + 1) % 20 == 0:
            print(f'Processed {i + 1}/{len(las_files)} files')

    except Exception as e:
        errors.append((las_file, str(e)))
        print(f'x Error in {las_file}: {e}')

print(f'\nSuccessfully processed {len(all_data)} files')
if errors:
    print(f'{len(errors)} files had errors!')

combined_df = pd.concat(all_data, ignore_index=True)
print(f'\nCombined shape: {combined_df.shape}')
print(f'Columns: {list(combined_df.columns)}')

Processed 20/118 files
Processed 40/118 files
Processed 60/118 files
Processed 80/118 files
Processed 100/118 files

Successfully processed 118 files

Combined shape: (2337639, 28)
Columns: ['FORCE_2020_LITHOFACIES_CONFIDENCE', 'FORCE_2020_LITHOFACIES_LITHOLOGY', 'CALI', 'MUDWEIGHT', 'ROP', 'RDEP', 'RSHA', 'RMED', 'RXO', 'SP', 'DTC', 'NPHI', 'PEF', 'GR', 'RHOB', 'DRHO', 'DEPTH_MD', 'X_LOC', 'Y_LOC', 'Z_LOC', 'WELL', 'DEPTH', 'BS', 'DCAL', 'SGR', 'ROPA', 'DTS', 'RMIC']


In [12]:
print('Unique values in FORCE_2020_LITHOFACIES_LITHOLOGY:')
print(combined_df['FORCE_2020_LITHOFACIES_LITHOLOGY'].unique()[:20])
print(f"\nTotal unique values: {combined_df['FORCE_2020_LITHOFACIES_LITHOLOGY'].nunique()}")
print(f"Missing values: {combined_df['FORCE_2020_LITHOFACIES_LITHOLOGY'].isna().sum()}")

print("\n\nAll columns with dtype 'object':")
print(combined_df.select_dtypes(include=['object']).columns.tolist())

Unique values in FORCE_2020_LITHOFACIES_LITHOLOGY:
[   nan 65000. 30000. 65030. 70000. 99000. 80000. 86000. 74000. 70032.
 90000. 88000. 93000.]

Total unique values: 12
Missing values: 906256


All columns with dtype 'object':
['WELL']


/var/folders/np/7jg1wp0n1bb1dhm8911t6pnm0000gn/T/ipykernel_6213/3613525317.py:7: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  print(combined_df.select_dtypes(include=['object']).columns.tolist())


In [13]:
print("Sample values from FORCE_2020_LITHOFACIES_LITHOLOGY:")
print(combined_df['FORCE_2020_LITHOFACIES_LITHOLOGY'].dropna().head(10))

print(f"\nUnique lithofacies: {combined_df['FORCE_2020_LITHOFACIES_LITHOLOGY'].nunique()}")
print(f"Missing: {combined_df['FORCE_2020_LITHOFACIES_LITHOLOGY'].isna().sum()}")

text_cols = combined_df.select_dtypes(include=['object']).columns.tolist()
print(f"\nText columns: {text_cols}")

Sample values from FORCE_2020_LITHOFACIES_LITHOLOGY:
3089    65000.0
3090    65000.0
3091    65000.0
3092    65000.0
3093    65000.0
3094    65000.0
3095    65000.0
3096    65000.0
3097    65000.0
3098    65000.0
Name: FORCE_2020_LITHOFACIES_LITHOLOGY, dtype: float64

Unique lithofacies: 12
Missing: 906256

Text columns: ['WELL']


/var/folders/np/7jg1wp0n1bb1dhm8911t6pnm0000gn/T/ipykernel_6213/3889833799.py:7: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  text_cols = combined_df.select_dtypes(include=['object']).columns.tolist()


In [15]:
print("Sample FORCE_2020_LITHOFACIES_LITHOLOGY values:")
sample = combined_df['FORCE_2020_LITHOFACIES_LITHOLOGY'].dropna().head(10)
print(sample.values)

print(f"\nUnique: {combined_df['FORCE_2020_LITHOFACIES_LITHOLOGY'].nunique()}")
print(f"Missing: {combined_df['FORCE_2020_LITHOFACIES_LITHOLOGY'].isna().sum()}")

print('\nColumn dtypes:')
print(combined_df.dtypes)

Sample FORCE_2020_LITHOFACIES_LITHOLOGY values:
[65000. 65000. 65000. 65000. 65000. 65000. 65000. 65000. 65000. 65000.]

Unique: 12
Missing: 906256

Column dtypes:
FORCE_2020_LITHOFACIES_CONFIDENCE    float64
FORCE_2020_LITHOFACIES_LITHOLOGY     float64
CALI                                 float64
MUDWEIGHT                            float64
ROP                                  float64
RDEP                                 float64
RSHA                                 float64
RMED                                 float64
RXO                                  float64
SP                                   float64
DTC                                  float64
NPHI                                 float64
PEF                                  float64
GR                                   float64
RHOB                                 float64
DRHO                                 float64
DEPTH_MD                             float64
X_LOC                                float64
Y_LOC                     

In [16]:
output_path = "/Users/wanbalqish/Projects/formation-top-prediction/data/raw/force_2020_combined.csv"

combined_df.to_csv(output_path, index=False)
print(f'Saved to: {output_path}')
print(f'File save: {os.path.getsize(output_path) / 1e6:.1f} MB')
print(f'\nDataframe info:')
print(f'Rows: {len(combined_df)}')
print(f'Columns: {len(combined_df.columns)}')
print(f' Memory: {combined_df.memory_usage(deep=True).sum() / 1e6:.1f} MB')

Saved to: /Users/wanbalqish/Projects/formation-top-prediction/data/raw/force_2020_combined.csv
File save: 475.9 MB

Dataframe info:
Rows: 2337639
Columns: 28
 Memory: 660.1 MB


In [17]:
df = pd.read_csv('/Users/wanbalqish/Projects/formation-top-prediction/data/raw/force_2020_combined.csv')

print('Dataset loaded')
print(f'Shape: {df.shape}')
print(f'\nFirst few rows:')
print(df.head())

print(f'\n\Basic statistics:')
print(df.describe())

Dataset loaded
Shape: (2337639, 28)

First few rows:
   FORCE_2020_LITHOFACIES_CONFIDENCE  FORCE_2020_LITHOFACIES_LITHOLOGY  CALI  \
0                                NaN                               NaN   NaN   
1                                NaN                               NaN   NaN   
2                                NaN                               NaN   NaN   
3                                NaN                               NaN   NaN   
4                                NaN                               NaN   NaN   

   MUDWEIGHT  ROP  RDEP  RSHA  RMED  RXO  SP  ...  Y_LOC  Z_LOC  \
0        NaN  NaN   NaN   NaN   NaN  NaN NaN  ...    NaN    NaN   
1        NaN  NaN   NaN   NaN   NaN  NaN NaN  ...    NaN    NaN   
2        NaN  NaN   NaN   NaN   NaN  NaN NaN  ...    NaN    NaN   
3        NaN  NaN   NaN   NaN   NaN  NaN NaN  ...    NaN    NaN   
4        NaN  NaN   NaN   NaN   NaN  NaN NaN  ...    NaN    NaN   

                         WELL   DEPTH  BS  DCAL  SGR  ROPA  DTS

In [20]:
print('Missing data summary:')
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(1)
missing_df = pd.DataFrame({
    'Column': df.columns,
    'Missing_Count': missing.values,
    'Missing_%': missing_pct.values
}).sort_values('Missing_%', ascending=False)

print(missing_df)

print(f"\n\nTarget variable (FORCE_2020_LITHOFACIES_LITHOLOGY):")
print(f"  Non-null: {df['FORCE_2020_LITHOFACIES_LITHOLOGY'].notna().sum():,}")
print(f"  Classes: {df['FORCE_2020_LITHOFACIES_LITHOLOGY'].nunique()}")

Missing data summary:
                               Column  Missing_Count  Missing_%
24                                SGR        2265476       96.9
27                               RMIC        2099262       89.8
23                               DCAL        1953401       83.6
8                                 RXO        1951432       83.5
26                                DTS        1948818       83.4
3                           MUDWEIGHT        1897939       81.2
25                               ROPA        1598886       68.4
6                                RSHA        1469179       62.8
12                                PEF        1413499       60.5
11                               NPHI        1304036       55.8
4                                 ROP        1116713       47.8
9                                  SP        1108869       47.4
15                               DRHO        1012729       43.3
14                               RHOB         941122       40.3
1    FORCE_2020_LI

In [21]:
# Remove rows with missing target
df_labeled = df[df['FORCE_2020_LITHOFACIES_LITHOLOGY'].notna()].copy()
print(f"Rows with target label: {len(df_labeled):,}")

# Select features based on completeness
# Keep columns with <50% missing
completeness = 1 - (df_labeled.isnull().sum() / len(df_labeled))
good_features = completeness[completeness > 0.5].index.tolist()

print(f"\nFeatures with >50% data completeness:")
for col in sorted(good_features):
    pct = completeness[col] * 100
    print(f"  {col}: {pct:.1f}%")

print(f"\nTotal usable features: {len(good_features)}")

Rows with target label: 1,431,383

Features with >50% data completeness:
  BS: 57.6%
  CALI: 93.2%
  DEPTH: 100.0%
  DEPTH_MD: 99.2%
  DRHO: 84.7%
  DTC: 93.9%
  FORCE_2020_LITHOFACIES_CONFIDENCE: 100.0%
  FORCE_2020_LITHOFACIES_LITHOLOGY: 100.0%
  GR: 99.9%
  NPHI: 67.6%
  PEF: 61.9%
  RDEP: 99.2%
  RHOB: 86.8%
  RMED: 96.5%
  SP: 68.4%
  WELL: 100.0%
  X_LOC: 99.2%
  Y_LOC: 99.2%
  Z_LOC: 99.2%

Total usable features: 19
